In [1]:
import pandas as pd
import json
import re
orders = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users_data = json.load(f)
users = pd.DataFrame(users_data)
restaurants_list = []
with open('restaurants.sql', 'r') as f:
    for line in f:
        if line.startswith('INSERT INTO restaurants VALUES'):
            match = re.search(r'\((.*)\);', line)
            if match:
                parts = [p.strip().strip("'") for p in match.group(1).split(',')]
                restaurants_list.append({
                    'restaurant_id': int(parts[0]),
                    'restaurant_name_sql': parts[1],
                    'cuisine': parts[2],
                    'rating': float(parts[3])
                })
restaurants = pd.DataFrame(restaurants_list)
merged_df = pd.merge(orders, users, on='user_id', how='left')
final_df = pd.merge(merged_df, restaurants, on='restaurant_id', how='left')
final_df.to_csv('final_food_delivery_dataset.csv', index=False)

# --- Analysis Examples (Answering the Hackathon Questions) ---

# Q: Gold membership orders
gold_orders = final_df[final_df['membership'] == 'Gold']
print(f"Total Gold Orders: {len(gold_orders)}")

# Q: Revenue from Hyderabad
hyd_rev = final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum()
print(f"Hyderabad Total Revenue: {round(hyd_rev)}")

# Q: Distinct users
print(f"Distinct Users: {final_df['user_id'].nunique()}")

# Q: Avg Order Value for Gold Members
print(f"Gold Avg Order Value: {round(gold_orders['total_amount'].mean(), 2)}")

Total Gold Orders: 4987
Hyderabad Total Revenue: 1889367
Distinct Users: 2883
Gold Avg Order Value: 797.15
